# Basic webscraping example

This notebook is an introductory exploration notebook. In this notebook, we look at the CSV provided by the course instructors, run their example scraping script, experiment with scraping the first 50 URLs, and split the URLs into CSVs for just Fox News and just NBC news (only wrote Fox News to CSV).

Since we have ~4000 urls to scrape, we should probably have some type of standard throttle with a retry/backoff for difficult links. Might be good to split the data into thirds and run separately to reduce the amount of time for each of us.

## Load data

Basic loading and looking at data. Data is a DF with 3000+ links in a `url` column.

In [2]:
import pandas as pd
links = pd.read_csv("../data/external/url_only_data.csv")

list(links["url"])

['https://www.foxnews.com/lifestyle/jack-carrs-eisenhower-d-days-memo-noble-undertaking',
 'https://www.foxnews.com/entertainment/bruce-willis-demi-moore-avoided-doing-one-thing-while-co-parenting-daughter-says',
 'https://www.foxnews.com/politics/blinken-meets-with-qatars-prime-minister.print',
 'https://www.foxnews.com/entertainment/emily-blunt-says-toes-curl-when-people-their-kids-want-act-want-say-dont-do-it',
 'https://www.foxnews.com/media/the-view-co-host-cnn-commentator-ana-navarro-host-night-2-democratic-national-convention',
 'https://www.foxnews.com/politics/striking-boeing-workers-boo-after-democrat-sen-maria-cantwell-criticizes-trump.print',
 'https://www.foxnews.com/politics/white-house-grilled-harris-gun-ownership-mandatory-gun-buybacks.print',
 'https://www.foxnews.com/media/tom-cotton-turns-tables-cnns-dana-bash-guns-brings-up-harris-past-remarks-school-shootings',
 'https://www.foxnews.com/politics/kamala-rides-tsunami-positive-press-skeptics-see-risky-choice',
 'http

## Description Example

Run example scraping code from project description document.

In [4]:
import requests
from bs4 import BeautifulSoup

url = "https://www.foxnews.com/sports/juan-soto-sends-yankees-world-series-first-time-15-years" # example website

response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Failed to load page: Status code {response.status_code}")

# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

title = soup.find("h1", class_="headline speakable")

# title should be the following
# Juan Soto sends the Yankees to the World Series for the first time in years

In [5]:
title.get_text()

'Juan Soto sends the Yankees to the World Series for the first time in 15 years'

## Scraping first 50 urls

Write function to get first 50 headlines from list of urls.

In [ ]:
# Get first 50 titles and store in an array -> do we need to
# throttle requests? it's not hitting the same link right?
import time
from requests.exceptions import RequestException
from collections import defaultdict
import random

def get_headlines(urls: list[str], verbose = False) -> dict[str, list[str]]:
    headlines = defaultdict(list)
    for url in urls:
        time.sleep(random.randint(1,3))
        print(url) if verbose else ""
        source = "fox" if "fox" in url else "nbc"
        
        try: 
            response = requests.get(url)
            response.raise_for_status()
        except RequestException as e:
            print(f"Error {e}. Trying again.")
            
            try:
                time.sleep(5)
                response = requests.get(url)
                response.raise_for_status()
            except:
                print(f"Error again: {e}. Skipping {url}")
                continue
        
        else:
            soup = BeautifulSoup(response.text, "html.parser")
            title = soup.find("h1", class_ = "headline speakable")
            headlines[source].append(title.get_text() if title else None)
        
    return headlines

In [7]:
urls = list(links["url"])[0:50]
headlines = get_headlines(urls = urls)

### Results

Display results and make sure all pages were successfully scraped. 

In [8]:
import pprint
pprint.pprint(headlines)

defaultdict(<class 'list'>,
            {'fox': ["Jack Carr recalls Gen. Eisenhower's D-Day memo about "
                     "'great and noble undertaking'",
                     'Bruce Willis, Demi Moore avoided doing one thing while '
                     'co-parenting, daughter says',
                     None,
                     'Emily Blunt says her ‘toes curl’ when people tell her '
                     "their kids want to act: 'I want to say, don’t do it!'",
                     "'The View' co-host, CNN commentator Ana Navarro to host "
                     'night 2 of Democratic National Convention',
                     None,
                     None,
                     "Tom Cotton turns tables on CNN's Dana Bash on guns, "
                     "brings up Harris' past remarks on school shootings",
                     'Kamala rides tsunami of positive press, but skeptics see '
                     'a risky choice',
                     "Harris' running mate faces renewed

In [10]:
len(headlines["fox"])

50

## Testing scraping workflow

Run through a full scraping process on first link in DF to make sure all selectors work. Code below is basically copy and pasted into `scripts/scrape_fox.py`.

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests

In [8]:
link = links["url"][0]
link

'https://www.foxnews.com/lifestyle/jack-carrs-eisenhower-d-days-memo-noble-undertaking'

In [9]:
# Fetch link content
response = requests.get(link)
# Check if request was successful
if response.status_code != 200:
    raise Exception(f"Failed to load page: Status code {response.status_code}")
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [11]:
# Show soup
print(soup.prettify())

<!DOCTYPE html>
<html data-n-head="%7B%22lang%22:%7B%22ssr%22:%22en%22%7D%7D" data-n-head-ssr="" lang="en">
 <head>
  <title>
   Jack Carr recalls Eisenhower's D-Day memo about 'great and noble undertaking' | Fox News
  </title>
  <meta content="IE=edge,chrome=1" data-n-head="ssr" http-equiv="X-UA-Compatible"/>
  <meta content="text/html; charset=utf-8" data-hid="content-type" data-n-head="ssr" http-equiv="content-type"/>
  <meta content="on" data-hid="x-dns-prefetch-control" data-n-head="ssr" http-equiv="x-dns-prefetch-control"/>
  <meta charset="utf-8" data-n-head="ssr"/>
  <meta content="width=device-width, minimum-scale=1.0, initial-scale=1.0" data-hid="viewport" data-n-head="ssr" name="viewport"/>
  <meta content="//static.foxnews.com/static/orion/styles/img/fox-news/favicons/mstile-70x70.png" data-n-head="ssr" name="msapplication-square70x70logo"/>
  <meta content="//static.foxnews.com/static/orion/styles/img/fox-news/favicons/mstile-150x150.png" data-n-head="ssr" name="msapplica

In [39]:
# Extract article topic, headline, subtitle, author, date posted
topic_span = soup.find("span", class_ = "eyebrow")
topic = topic_span.get_text() if topic_span else None

title_h1 = soup.find("h1", class_ = "headline speakable")
title = title_h1.get_text() if title_h1 else None

subtitle_h2 = soup.find("h2", class_ = "sub-headline speakable")
subtitle = subtitle_h2.get_text() if subtitle_h2 else None

# Author
author_div = soup.find("div", class_ = "author-byline")
author = author_div.find_all("a")[0].get_text() if author_div else None

# Date posted
date_posted_time= soup.find("time")
datetime_posted = date_posted_time.get("datetime") if date_posted_time else None

In [42]:
# create dict
article_info = {
    "url": link,
    "topic": topic,
    "title": title,
    "subtitle": subtitle,
    "author": author,
    "datetime_posted": datetime_posted,
    "label": "Fox News"
}

## Splitting links into Fox News and NBC News dfs

Since we split up scraping responsibilities by news outlet, this section just splits the links df by it's news outlet url. There are also some cells confirming that we properly split the data between NBC and Fox.

In [4]:
links

,url
0,https://www.foxnews.com/lifestyle/jack-carrs-e...
1,https://www.foxnews.com/entertainment/bruce-wi...
2,https://www.foxnews.com/politics/blinken-meets...
3,https://www.foxnews.com/entertainment/emily-bl...
4,https://www.foxnews.com/media/the-view-co-host...
...,...
3800,https://www.nbcnews.com/politics/2024-election...
3801,https://www.nbcnews.com/select/shopping/best-a...
3802,https://www.nbcnews.com/select/shopping/best-v...
3803,https://www.nbcnews.com/politics/2024-election...


In [21]:
fox_links = links[links["url"].str.contains("foxnews.com")]

In [22]:
fox_links[fox_links["url"].str.contains("nbcnews")]

,url


In [23]:
fox_links

,url
0,https://www.foxnews.com/lifestyle/jack-carrs-e...
1,https://www.foxnews.com/entertainment/bruce-wi...
2,https://www.foxnews.com/politics/blinken-meets...
3,https://www.foxnews.com/entertainment/emily-bl...
4,https://www.foxnews.com/media/the-view-co-host...
...,...
1995,https://www.foxnews.com/official-polls/fox-new...
1996,https://www.foxnews.com/politics/trump-harris-...
1997,https://www.foxnews.com/world/biden-admin-outl...
1998,https://www.foxnews.com/politics/kamala-harris...


In [ ]:
fox_links.to_csv("../data/external/fox_news_urls.csv", index = False) # Writing just Fox to CSV bc idk how Joe did his part yet.

In [24]:
nbc_news = links[~links["url"].str.contains("foxnews.com")]